In [1]:
import emoji
import nltk
import numpy as np
import pandas as pd
import re

from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

Change the path for the csv ***HERE*** accordingly to load/save

In [2]:
IN_CSV = "../../data/twitter_training.csv"
OUT_CSV = '../../data/output_file.csv'

TITLES = ["id", "entity", "sentiment", "text"]

In [3]:
df = pd.read_csv(IN_CSV, names=TITLES).astype(str)

In [4]:
df.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [5]:
tokenizer = TweetTokenizer()

Bingxi: Note for myself that twitter usernames are not case sensitive.

In [6]:
def load_emoticon_dict(dict_pth="../../data/emoji/emoticon.txt"):
    emoticon_dict = {}

    with open(dict_pth, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f]

    i = 0
    while i < len(lines):
        if lines[i] == "":
            i += 1
            continue

        emoticons = lines[i].split()
        meaning = lines[i + 1].upper()
        
        meaning = re.sub(r",", " ", meaning)
        meaning = re.sub(r"\s+", "_", meaning)
        
        for emo in emoticons:
            emo = re.sub(r"&nbsp;", " ", emo)
            emoticon_dict[emo] = meaning

        i += 3

    return emoticon_dict

In [7]:
emoticon_dict = load_emoticon_dict()

In [8]:
print(emoticon_dict)

{':‑)': 'SMILEY', ':)': 'SMILEY', ':-]': 'SMILEY', ':]': 'SMILEY', ':->': 'SMILEY', ':>': 'SMILEY', '8-)': 'SMILEY', '8)': 'SMILEY', ':-}': 'SMILEY', ':}': 'SMILEY', ':^)': 'SMILEY', '=]': 'SMILEY', '=)': 'SMILEY', ':‑D': 'BIG_GRIN', ':D': 'BIG_GRIN', '8‑D': 'BIG_GRIN', '8D': 'BIG_GRIN', '=D': 'BIG_GRIN', '=3': 'PLAYFUL_CUTE_CAT', 'B^D': 'BIG_GRIN', 'c:': 'BIG_GRIN', 'C:': 'BIG_GRIN', 'x‑D': 'LAUGHING', 'xD': 'LAUGHING', 'X‑D': 'LAUGHING', 'XD': 'LAUGHING', ':-))': 'DOUBLE_CHIN', ':))': 'DOUBLE_CHIN', ':‑(': 'FROWN_SAD', ':(': 'FROWN_SAD', ':‑c': 'FROWN_SAD', ':c': 'FROWN_SAD', ':‑<': 'FROWN_SAD', ':<': 'FROWN_SAD', ':‑[': 'FROWN_SAD', ':[': 'FROWN_SAD', ':-||': 'FROWN_SAD', ':{': 'FROWN_SAD', ':@': 'FROWN_SAD', ';(': 'FROWN_SAD', ":'‑(": 'CRYING', ":'(": 'CRYING', ':=(': 'CRYING', ":'‑)": 'TEARS_OF_HAPPINESS', ":')": 'TEARS_OF_HAPPINESS', ':"D': 'TEARS_OF_HAPPINESS', '>:(': 'ANGRY', '>:[': 'ANGRY', "D‑':": 'HORROR', 'D:<': 'DISGUST', 'D;': 'SADNESS', 'D:': 'DISMAY', 'D8': 'DISMAY', 'D

In [14]:
def preprocess(row):
    entity = row["entity"]
    text = row["text"]

    FLAG = re.I

    emoji_pattern = r"(\ud83c[d000-dfff]|\ud83d[d000-dfff]|\ud83e[d000-dfff])"

    url_pattern = r'(https?:\/\/)?(([\w\-]{1,63}\.)+[\w\-]{1,63})(\/[\w.\-]+)*([?=+%\w~\-.&]*)'
    entity_pattern = rf"@?{entity}"
    user_pattern = r"@\w+"
    repeated_pattern = r"(.)\1\1\1+"
    repeated_replace_pattern = r"\1\1\1"
    
    # Concate if potential URL/Username
    text = re.sub(r"\s+\/(\s+)?", "/", text)
    text = re.sub(r"\@\s+", "@", text)

    # Replaces URL
    text = re.sub(url_pattern, '<URL>', text)

    # Replaces entity and other_user
    text = re.sub(entity_pattern, "<ENTITY>", text, flags=FLAG)
    text = re.sub(user_pattern, "<OTHER_USER>", text)

    # Raplaces repeated patterns
    text = re.sub(repeated_pattern, repeated_replace_pattern, text)
    
    tokens = tokenizer.tokenize(text)

    for i, token in enumerate(tokens):
        # Replaces emoji
        if emoji.is_emoji(token):
            print(f"{row.name}: {token}")
            meaning = emoji.demojize(token)[1:-1]
            emoji_replacement = f"<EMOJI_{meaning.upper()}>"
            print(f"{token} -> {emoji_replacement}")
            tokens[i] = emoji_replacement
        elif token in emoticon_dict and len(token) > 1:
            print(f"==Row {row.name}== {token}")
            meaning = emoticon_dict[token]
            emoticon_replacement = f"<EMOTICON_{meaning}>"
            print(f"{token} -> {emoticon_replacement}")
            tokens[i] = emoticon_replacement
        else:
            continue
    
    return tokens

In [15]:
df["cleaned_text"] = df.apply(preprocess, axis=1)

==Row 6== :)
:) -> <EMOTICON_SMILEY>
==Row 7== :)
:) -> <EMOTICON_SMILEY>
==Row 9== :)
:) -> <EMOTICON_SMILEY>
==Row 10== :)
:) -> <EMOTICON_SMILEY>
78: ‼
‼ -> <EMOJI_DOUBLE_EXCLAMATION_MARK>
81: ‼
‼ -> <EMOJI_DOUBLE_EXCLAMATION_MARK>
==Row 120== :)
:) -> <EMOTICON_SMILEY>
==Row 121== :)
:) -> <EMOTICON_SMILEY>
==Row 122== :)
:) -> <EMOTICON_SMILEY>
==Row 123== :)
:) -> <EMOTICON_SMILEY>
==Row 125== :)
:) -> <EMOTICON_SMILEY>
222: 1⃣
1⃣ -> <EMOJI_KEYCAP_1>
225: 1⃣
1⃣ -> <EMOJI_KEYCAP_1>
==Row 289== :/
:/ -> <EMOTICON_SKEPTICAL_ANNOYED_UNEASY_HESITANT>
==Row 385== :/
:/ -> <EMOTICON_SKEPTICAL_ANNOYED_UNEASY_HESITANT>
==Row 535== :/
:/ -> <EMOTICON_SKEPTICAL_ANNOYED_UNEASY_HESITANT>
==Row 536== :/
:/ -> <EMOTICON_SKEPTICAL_ANNOYED_UNEASY_HESITANT>
576: 🤔
🤔 -> <EMOJI_THINKING_FACE>
579: 🤔
🤔 -> <EMOJI_THINKING_FACE>
==Row 582== </3
</3 -> <EMOTICON_BROKEN_HEART>
==Row 583== </3
</3 -> <EMOTICON_BROKEN_HEART>
==Row 585== </3
</3 -> <EMOTICON_BROKEN_HEART>
==Row 586== </3
</3 -> <EMOTICON_BR

Bingxi: 
    This is for me to test whether my code for replacing emoji is working

In [16]:
df.iloc[576]

id                                                           2500
entity                                                Borderlands
sentiment                                              Irrelevant
text            Top 4 favourite games you say? 🤔. . Sea of Thi...
cleaned_text    [Top, 4, favourite, games, you, say, ?, <EMOJI...
Name: 576, dtype: object

In [17]:
df.head()

,id,entity,sentiment,text,cleaned_text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,"[im, getting, on, <ENTITY>, and, i, will, murd..."
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,"[I, am, coming, to, the, borders, and, I, will..."
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,"[im, getting, on, <ENTITY>, and, i, will, kill..."
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,"[im, coming, on, <ENTITY>, and, i, will, murde..."
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,"[im, getting, on, <ENTITY>, 2, and, i, will, m..."


In [13]:
df.to_csv(OUT_CSV)